# Day 13：同一世界下的 Year 2 过程模块

这一讲演示 ChemWorld 如何在同一个 `WorldLaw` 下开放结晶、蒸馏、连续流和电化学任务。它们不是新的小游戏，而是同一个物理化学世界里的不同 task slice。

## 学习路径定位

| 项目 | 内容 |
| --- | --- |
| 阶段 | E. 研究延展 |
| 难度 | 研究延展 |
| 先修 | 理解统一 world law、task slice 和 reaction-to-purification。 |
| 今天只解决 | 观察结晶、蒸馏、连续流、电化学如何作为同一世界下的过程模块扩展。 |
| 今天不要求 | 不宣称这些模块已经达到专业过程模拟器精度。 |
| 本日交付 | 一个跨过程 task 的最小闭环、一个需要新增的物理账本、一个后续开发计划。 |
| 下一步如何复用 | 这些产出会进入专业级 TODO 和后续环境设计。 |



## 课堂时间盒：每 30 分钟都有产出

建议按 3 小时工作坊使用。每一段都要留下一个小证据，不要只运行代码看到结果就继续往下翻。

| 时间 | 阶段目标 | 具体动作 | 当段产出 |
| --- | --- | --- | --- |
| 0:00-0:30 | 查看扩展模块 | 读取 WorldLawSpec 中的结晶、蒸馏、连续流、电化学模块。 | 确认它们共享同一 world law。 |
| 0:30-1:00 | 运行结晶任务 | 执行 scripted task 并读取 crystal 指标。 | 解释纯度/收率权衡。 |
| 1:00-1:30 | 运行蒸馏任务 | 执行蒸馏 task 并读取 distillate 指标。 | 解释能耗/回收权衡。 |
| 1:30-2:00 | 运行连续流/电化学 | 比较 flow_conversion、selectivity、energy_efficiency。 | 得到跨过程对比表。 |
| 2:00-2:30 | 检查前置条件 | 用 validator 暴露状态依赖动作。 | 说明为什么不是小游戏拼接。 |
| 2:30-3:00 | 设计新 task | 基于共享操作语言提出一个跨过程 task。 | 写出 task card 草案。 |

教师提示：如果课堂时间少于 3 小时，可以把最后两个时间盒改成课后提交；但前四个时间盒建议现场完成。


## 学习目标

- 查看 Year 2 过程模块是否进入统一 `WorldLawSpec`。
- 使用 scripted baseline 跑通四个新增 task。
- 汇总 final assay 中的过程指标。
- 用 validator 理解状态依赖的动作前置条件。

## 本日任务梯度

| 层级 | 任务 |
| --- | --- |
| 基础任务 | 运行一个 Year 2 task 并记录新增观测字段。 |
| 进阶任务 | 比较不同过程模块的 score、risk、cost 和 mass-balance 字段。 |
| 挑战任务 | 为其中一个模块写出下一步要实现的专业物理模型清单。 |
| 反思问题 | 为什么这些过程不应该做成独立小游戏，而应共享同一套 WorldLaw？ |



In [1]:
from __future__ import annotations

import gymnasium as gym
import pandas as pd

import chemworld  # noqa: F401
from chemworld.agents.base import HistoryRecord
from chemworld.agents.event import ScriptedChemistryAgent
from chemworld.tasks import get_task
from chemworld.world import world_law_spec
from chemworld.wrappers import ActionMaskWrapper, validate_event_action


## 1. 查看统一世界规律

`WorldLawSpec` 记录所有模块。新增过程应出现在同一个 world law 里，而不是注册成独立环境。

In [2]:
law = world_law_spec().to_dict()
modules = pd.DataFrame(law["ontology_registry"]["modules"])
modules[["module_id", "version"]]

,module_id,version
0,reaction,0.3
1,thermal,0.3
2,phase_partition,0.3
3,separation,0.3
4,crystallization,0.1
5,distillation,0.1
6,continuous_flow,0.1
7,electrochemistry,0.1
8,instrument_observation,0.3


## 2. 运行四个 Year 2 task

下面用同一个 scripted baseline 运行四个任务，并抓取每个任务最后一次 final assay。

In [3]:
PROCESS_TASKS = [
    "reaction-to-crystallization",
    "reaction-to-distillation",
    "flow-reaction-optimization",
    "electrochemical-conversion",
]

def run_scripted_task(task_id: str, seed: int = 0) -> dict[str, float]:
    task = get_task(task_id)
    env = gym.make("ChemWorld", task_id=task_id, seed=seed)
    obs, info = env.reset(seed=seed)
    agent = ScriptedChemistryAgent()
    agent.reset(info, seed=seed)
    history: list[HistoryRecord] = []
    last_assay = None
    final_count = 0
    invalid_count = 0
    for _ in range(task.budget):
        action = agent.act(history)
        obs, reward, terminated, truncated, step_info = env.step(action)
        history.append(
            HistoryRecord(
                step=len(history) + 1,
                action=action,
                observation=obs,
                reward=reward,
                info=step_info,
            )
        )
        flags = step_info.get("constraint_flags", {})
        invalid_count += int(flags.get("precondition_failed", False))
        if action.get("operation") == "measure" and action.get("instrument") == "final_assay":
            final_count += 1
            last_assay = obs
        if terminated or truncated:
            break
    env.close()
    assay = last_assay or obs
    keys = [
        "score",
        "yield",
        "crystal_yield",
        "crystal_purity",
        "distillate_purity",
        "distillate_recovery",
        "flow_conversion",
        "electrochemical_selectivity",
        "energy_efficiency",
        "safety_risk",
        "cost",
    ]
    row = {"task_id": task_id, "steps": len(history), "final_assay_count": final_count, "invalid_count": invalid_count}
    for key in keys:
        value = assay.get(key)
        row[key] = float(value[0]) if value is not None and value[0] == value[0] else None
    return row

results = pd.DataFrame([run_scripted_task(task_id) for task_id in PROCESS_TASKS])
results

,task_id,steps,final_assay_count,invalid_count,score,yield,crystal_yield,crystal_purity,distillate_purity,distillate_recovery,flow_conversion,electrochemical_selectivity,energy_efficiency,safety_risk,cost
0,reaction-to-crystallization,15,1,0,0.000000,0.037522,0.236729,0.180611,0.003554,0.000000,0.00000,0.009408,0.023895,0.094769,0.703707
1,reaction-to-distillation,15,1,0,0.125884,0.236240,0.000000,0.002202,0.598493,0.564341,0.00649,0.002576,0.005686,0.168179,0.724257
2,flow-reaction-optimization,60,7,0,0.474624,0.637453,0.006132,0.000000,0.010289,0.000000,0.99712,0.000000,0.000000,0.144886,0.441180
3,electrochemical-conversion,48,6,0,0.168455,0.069217,0.000000,0.010802,0.000835,0.000000,0.00000,0.850862,0.825472,0.081148,0.260261


## 3. 对比过程指标

不同 task 的主指标不同，所以不能把所有任务混成一个总榜。这里先做可视化检查：每个任务是否产生了自己关心的过程指标。

In [4]:
plot_cols = [
    "crystal_purity",
    "distillate_purity",
    "flow_conversion",
    "electrochemical_selectivity",
    "energy_efficiency",
]

def bar(value: float | None, width: int = 24) -> str:
    if value is None:
        return "未观测"
    filled = round(max(0.0, min(1.0, value)) * width)
    return "█" * filled + "░" * (width - filled) + f" {value:.3f}"

visual = results[["task_id", *plot_cols]].copy()
for col in plot_cols:
    visual[col] = visual[col].map(bar)
visual

,task_id,crystal_purity,distillate_purity,flow_conversion,electrochemical_selectivity,energy_efficiency
0,reaction-to-crystallization,████░░░░░░░░░░░░░░░░░░░░ 0.181,░░░░░░░░░░░░░░░░░░░░░░░░ 0.004,░░░░░░░░░░░░░░░░░░░░░░░░ 0.000,░░░░░░░░░░░░░░░░░░░░░░░░ 0.009,█░░░░░░░░░░░░░░░░░░░░░░░ 0.024
1,reaction-to-distillation,░░░░░░░░░░░░░░░░░░░░░░░░ 0.002,██████████████░░░░░░░░░░ 0.598,░░░░░░░░░░░░░░░░░░░░░░░░ 0.006,░░░░░░░░░░░░░░░░░░░░░░░░ 0.003,░░░░░░░░░░░░░░░░░░░░░░░░ 0.006
2,flow-reaction-optimization,░░░░░░░░░░░░░░░░░░░░░░░░ 0.000,░░░░░░░░░░░░░░░░░░░░░░░░ 0.010,████████████████████████ 0.997,░░░░░░░░░░░░░░░░░░░░░░░░ 0.000,░░░░░░░░░░░░░░░░░░░░░░░░ 0.000
3,electrochemical-conversion,░░░░░░░░░░░░░░░░░░░░░░░░ 0.011,░░░░░░░░░░░░░░░░░░░░░░░░ 0.001,░░░░░░░░░░░░░░░░░░░░░░░░ 0.000,████████████████████░░░░ 0.851,████████████████████░░░░ 0.825


## 4. 状态依赖前置条件

物理世界不能允许“没结晶就过滤”。`ActionMaskWrapper` 和 `validate_event_action` 会把这种错误提前暴露给学生或 LLM-agent。

In [5]:
env = ActionMaskWrapper(gym.make("ChemWorld", task_id="reaction-to-crystallization", seed=0))
obs, info = env.reset(seed=0)

bad_filter = validate_event_action({"operation": "filter_crystals"}, env)
bad_filter["valid"], bad_filter["invalid_reasons"]

(False, ['has_volume', 'has_material', 'filter_requires_crystallization'])

In [6]:
for action in [
    {"operation": "add_solvent", "volume_L": 0.028, "solvent": 2},
    {"operation": "add_reagent", "amount_mol": 0.010},
    {"operation": "seed_crystals", "seed_mass_g": 0.006},
    {"operation": "cool_crystallize", "target_temperature_K": 278.15, "duration_s": 1200.0},
]:
    obs, reward, terminated, truncated, info = env.step(action)

good_filter = validate_event_action({"operation": "filter_crystals"}, env)
env.close()
good_filter["valid"], good_filter["invalid_reasons"]

(True, [])

## 小结

Year 2 的关键不是“多几个环境”，而是同一个物理化学世界逐步开放更多操作。agent 如果能学到反应、分离、结晶、蒸馏、流动和电化学之间共享的结构，它才是在学习局部 world model，而不是刷单个小游戏。